# Phase 0 · Environment and continuity checks (R1 revision)

This notebook verifies that the workstation can reproduce the submitted setting before any new experiment is run.
It does **not** train or change anything. Run it cell by cell and paste the final summary cell output back.

**Exit criteria (all must PASS):**

| Check | What is verified |
|---|---|
| C1 | Environment: GPU visible, pinned package versions installed |
| C2 | Dataset: RSNA labels and DICOM files complete, images are 1024 x 1024 |
| C3 | Split: the v1 test split is reproduced exactly (5,337 patients, 1,229 positive), identical to the patients in the v1 CSV (D-05) |
| C4 | Model A: the submitted weights load and their test probabilities are compared with `Model_Prob` in the v1 CSV (D-06) |
| C5 | v1 legacy CSV: hash and row count recorded (ART-001) |

## Configuration cell

All paths and constants come from `configs/config.yaml`. The notebook lives in `notebooks/`, so the repository root is its parent folder.
Every check writes its result into the `RESULTS` dictionary, which is saved at the end to `output/phase0/phase0_checks.json`.

In [ ]:
import os, sys, json, random, hashlib, platform
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
P = {k: (ROOT / v if not str(v).startswith("/") else Path(v)) for k, v in CFG["paths"].items() if k != "drive_backup_dir"}

SEED = CFG["project"]["seed"]
random.seed(SEED); np.random.seed(SEED)

OUT = P["phase0_out"]; OUT.mkdir(parents=True, exist_ok=True)
RESULTS = {}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def record(check, passed, **info):
    RESULTS[check] = {"pass": bool(passed), **info}
    print(f"[{check}] {'PASS' if passed else 'FAIL'}")
    for k, v in info.items():
        print(f"   {k}: {v}")

print("ROOT:", ROOT)
for k, v in P.items():
    print(f"{k:18s} {v}  exists={v.exists()}")

## C1 · Environment

Checks that the GPU is visible from WSL and that the installed versions match `requirements-r1.txt` (D-18).
A version mismatch is reported as FAIL because results must be reproducible with the pinned environment.

In [ ]:
import torch, torchvision, scipy, skimage, sklearn, cv2, pydicom, statsmodels
from importlib.metadata import version

expected = {"torch": "2.6.0", "torchvision": "0.21.0", "numpy": "1.26.4", "pandas": "2.2.3",
            "scipy": "1.14.1", "scikit-image": "0.24.0", "scikit-learn": "1.5.2",
            "opencv-python-headless": "4.10.0.84", "pydicom": "2.4.4", "grad-cam": "1.5.7",
            "statsmodels": "0.14.4"}
installed = {}
for pkg in expected:
    try:
        installed[pkg] = version(pkg).split("+")[0]
    except Exception:
        installed[pkg] = "NOT INSTALLED"
mismatch = {k: (installed[k], v) for k, v in expected.items() if installed[k] != v}

gpu = torch.cuda.is_available()
record("C1_environment", gpu and not mismatch,
       python=platform.python_version(),
       gpu=torch.cuda.get_device_name(0) if gpu else "NONE",
       cuda=torch.version.cuda,
       mismatched_versions=mismatch or "none")

## C2 · Dataset integrity

Checks the RSNA training labels and images:
- number of label rows and unique patients,
- that every patient has a single consistent `Target`,
- that a DICOM file exists for every patient,
- that a random sample of 50 DICOMs is 1024 x 1024 (the v1 code hard-codes this size when scaling the boxes).

In [ ]:
labels = pd.read_csv(P["labels_csv"])
n_rows = len(labels)
patients = labels["patientId"].unique()
target_per_patient = labels.groupby("patientId")["Target"].nunique()
inconsistent = int((target_per_patient > 1).sum())
n_pos_patients = int(labels.groupby("patientId")["Target"].max().sum())

files = set(os.listdir(P["image_dir"]))
missing = [pid for pid in patients if f"{pid}.dcm" not in files]

rng = np.random.default_rng(SEED)
sample = rng.choice(patients, size=min(50, len(patients)), replace=False)
shapes = {pydicom.dcmread(P["image_dir"] / f"{pid}.dcm").pixel_array.shape for pid in sample}

record("C2_dataset",
       inconsistent == 0 and not missing and shapes == {(1024, 1024)},
       label_rows=n_rows, unique_patients=len(patients), positive_patients=n_pos_patients,
       inconsistent_targets=inconsistent, missing_dicoms=len(missing),
       sampled_shapes=sorted(shapes),
       labels_sha256=sha256(P["labels_csv"]))

## C3 · Reproduce the v1 test split (D-05)

The v1 code (`main.py`, lines 73-85) builds the patient list in the order of `df_labels['patientId'].unique()`, keeps patients with an image file, and splits with `train_test_split(test_size=0.2, random_state=42)`.
The same logic is applied here and the resulting test set is compared with the patient IDs and labels in the v1 CSV.
If it matches, the split is saved as the frozen canonical split for every later phase (`output/phase0/split_train.csv`, `output/phase0/split_test.csv`).

In [ ]:
from sklearn.model_selection import train_test_split

valid_patients = [pid for pid in labels["patientId"].unique() if f"{pid}.dcm" in files]
train_ids, test_ids = train_test_split(valid_patients, test_size=CFG["split"]["test_size"],
                                       random_state=CFG["split"]["random_state"])

target_first = labels.drop_duplicates("patientId").set_index("patientId")["Target"]  # v1 uses .iloc[0]
test_pos = int(target_first.loc[test_ids].sum())

v1 = pd.read_csv(P["v1_legacy_csv"], usecols=["Patient", "Label", "Model_Prob"])
v1_pat = v1.drop_duplicates("Patient").set_index("Patient")
same_set = set(test_ids) == set(v1_pat.index)
label_agree = bool((target_first.loc[v1_pat.index].values == v1_pat["Label"].values).all()) if same_set else False
v1_order = list(dict.fromkeys(v1["Patient"]))
same_order = v1_order == list(test_ids)

if same_set:
    pd.DataFrame({"patientId": train_ids, "Target": target_first.loc[train_ids].values}).to_csv(OUT / "split_train.csv", index=False)
    pd.DataFrame({"patientId": test_ids, "Target": target_first.loc[test_ids].values}).to_csv(OUT / "split_test.csv", index=False)

record("C3_split",
       same_set and label_agree and len(test_ids) == CFG["split"]["expected_test_patients"]
       and test_pos == CFG["split"]["expected_test_positives"],
       n_train=len(train_ids), n_test=len(test_ids), test_positives=test_pos,
       identical_to_v1_csv=same_set, labels_agree_with_v1=label_agree, same_order_as_v1=same_order,
       split_test_sha256=sha256(OUT / "split_test.csv") if same_set else None)

## C4 · Model A weights and test probabilities (D-06)

Loads the submitted weights and runs the clean test images through the model with the exact v1 preprocessing:
DICOM, min-max to 0-255, grayscale to RGB, resize to 224 x 224, `ToTensor()`, **no** ImageNet normalization.

The probabilities are compared with `Model_Prob` in the v1 CSV:
- a maximum absolute difference near zero means these weights produced the v1 CSV;
- a large difference means the v1 CSV came from different weights (this is recorded in DEADENDS/LOG, not hidden).

Classifier metrics on the test set are also printed (they will be refined with CIs in Phase 1).

In [ ]:
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

w_path = P["model_a_weights"]
w_hash = sha256(w_path)

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
state = torch.load(w_path, map_location="cpu", weights_only=True)
load_info = model.load_state_dict(state, strict=True)
model = model.to(device).eval()

tf = T.Compose([T.ToPILImage(), T.Resize((CFG["image"]["size"],) * 2), T.ToTensor()])

class TestSet(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        pid = self.ids[i]
        img = pydicom.dcmread(P["image_dir"] / f"{pid}.dcm").pixel_array
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        return tf(img), pid

loader = DataLoader(TestSet(list(test_ids)), batch_size=64, shuffle=False, num_workers=8, pin_memory=True)
probs, pids = [], []
with torch.no_grad():
    for x, pid in loader:
        probs.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu().numpy()); pids += list(pid)
probs = pd.Series(np.concatenate(probs), index=pids, name="prob_A")
probs.to_csv(OUT / "modelA_test_probs.csv")

diff = (probs - v1_pat["Model_Prob"].reindex(probs.index)).abs()
y = target_first.loc[probs.index].values; yhat = (probs.values > 0.5).astype(int)
metrics = dict(auc=roc_auc_score(y, probs.values), acc=accuracy_score(y, yhat), precision=precision_score(y, yhat),
               recall=recall_score(y, yhat), f1=f1_score(y, yhat))

record("C4_modelA", diff.max() < 1e-3,
       weights_sha256=w_hash,
       max_abs_diff_vs_v1=float(diff.max()), mean_abs_diff_vs_v1=float(diff.mean()),
       n_diff_gt_1e3=int((diff > 1e-3).sum()),
       **{k: round(float(v), 4) for k, v in metrics.items()})

## C5 · v1 legacy CSV (ART-001)

Records the hash and row count of the v1 results CSV so that every later comparison refers to exactly this file.

In [ ]:
v1_rows = sum(1 for _ in open(P["v1_legacy_csv"])) - 1
record("C5_v1_legacy", v1_rows == 192132,
       rows=v1_rows, sha256=sha256(P["v1_legacy_csv"]))

## Summary and LOG entry

Saves all results to `output/phase0/phase0_checks.json` and prints a LOG entry ready to append to `_state/LOG.md`.
Paste this cell's output back into the chat.

In [ ]:
from datetime import date
(OUT / "phase0_checks.json").write_text(json.dumps(RESULTS, indent=2, default=str))

print("PHASE 0 SUMMARY")
for k, v in RESULTS.items():
    print(f"  {k:16s} {'PASS' if v['pass'] else 'FAIL'}")
all_pass = all(v["pass"] for v in RESULTS.values())
print("ALL PASS:", all_pass)

print("\n--- LOG entry (append to _state/LOG.md) ---")
print(f"## {date.today().isoformat()} | Phase 0 checks executed")
for k, v in RESULTS.items():
    info = {kk: vv for kk, vv in v.items() if kk != "pass"}
    print(f"- {k}: {'PASS' if v['pass'] else 'FAIL'} | {info}")
print(f"- Exit criteria met: {all_pass}")